# 02b – Rozšírený baseline (FRD)

Pokračovanie notebooku 02. Čistenie je rovnaké, rozdelenie pre seed 42 je rovnaké ako v 02.

- varianty textu, ktoré postupne odstraňujú artefakty generovania:
  - `raw`: pôvodný text
  - `norm`: medzera v spojení viet (`one.I` -> `one. I`), zlúčenie bielych znakov (dvojité medzery, nové riadky)
  - `trim`: `norm` + orezanie textu po poslednú ukončenú vetu (v oboch triedach); texty bez ukončenej vety sa vyhodia
  - `trim_len`: `trim` + zarovnanie rozdelenia dĺžky (podvzorkovanie väčšej triedy v pásmach dĺžky)
- pre každý variant: model len z artefaktov (kontrola, či artefakt zmizol), slovné a znakové TF-IDF + LR
- viac seedov (priemer ± std), leave-one-category-out (LOCO) ako test na neznámej doméne
- analýza modelu: top príznaky, false negatives, false positives

Vstup modelu: iba text.

In [1]:
from pathlib import Path

DATA_PATH = Path("../../data/FRD/fake reviews dataset.csv")
RESULTS = Path("../../results")
COL = {"text": "text_", "category": "category", "label": "label"}
LABEL_MAP = {"OR": 0, "CG": 1}

SEEDS = [42, 43, 44]       # 42 = rovnaké rozdelenie ako v 02
VAL_FRAC = 0.15
TEST_FRAC = 0.15
DEDUP = True

VARIANTS = ["raw", "norm", "trim", "trim_len"]
LEN_BINS = 20              # počet pásiem dĺžky pre trim_len (kvantily)
TARGET_FPR = 0.01

RUN_LOCO = True
LOCO_SEED = 42
LOCO_CHAR_VARIANTS = ["raw", "trim_len"]   # znakové modely v LOCO len pre krajné varianty (čas behu)
TOP_K = 25
N_EXAMPLES = 5

In [2]:
import re, time
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 300)

df = pd.read_csv(DATA_PATH)
txt = COL["text"]
df["y"] = df[COL["label"]].map(LABEL_MAP)
df["cat"] = df[COL["category"]].str.replace(r"_5$", "", regex=True)
df[txt] = df[txt].astype(str).str.strip()
df = df[df[txt] != ""]
if DEDUP:
    df = df.drop_duplicates(subset=[txt])
df = df.reset_index(drop=True)
CATS = sorted(df["cat"].unique())
print(len(df), df["y"].value_counts().to_dict())

40410 {0: 20215, 1: 20195}


## 1. Varianty textu
- `norm`: `([a-z0-9)][.!?])([A-Z])` -> vloženie medzery; `\s+` -> jedna medzera
- `trim`: orezanie za posledný znak `.`, `!`, `?` (prípadne s úvodzovkou / zátvorkou); text bez takého znaku sa vyhodí
- orezanie sa robí v oboch triedach rovnako -> pravé recenzie, ktoré nekončia interpunkciou, tiež stratia neukončený koniec

In [3]:
JOIN_RE = re.compile(r"([a-z0-9)][.!?])([A-Z])")
END_RE = re.compile(r"[.!?\"')\]]\s*$")
LAST_SENT_RE = re.compile(r"^(.*[.!?][\"')\]]*)", re.S)

def normalize(s):
    s = JOIN_RE.sub(r"\1 \2", s)
    return re.sub(r"\s+", " ", s).strip()

def trim(s):
    m = LAST_SENT_RE.match(s)
    return m.group(1).strip() if m else None

df["text_raw"] = df[txt]
df["text_norm"] = df["text_raw"].map(normalize)
df["text_trim"] = df["text_norm"].map(trim)

has_sent = df["text_trim"].notna()
print("texty bez ukončenej vety (vyhodené vo variantoch trim, trim_len):")
print(df[~has_sent].groupby("y").size().rename({0: "OR", 1: "CG"}).to_dict())
for v, col in [("raw", "text_raw"), ("norm", "text_norm"), ("trim", "text_trim")]:
    s = df[col].dropna()
    print(f"{v:5s} priemerne slov: OR {s[df['y']==0].str.split().str.len().mean():.1f}, "
          f"CG {s[df['y']==1].str.split().str.len().mean():.1f}")

ex = df[(df["y"] == 1) & df["text_raw"].str.contains(r"[a-z][.!?][A-Z]") & has_sent].iloc[0]
print("\nraw: ", repr(ex["text_raw"][-200:]))
print("norm:", repr(ex["text_norm"][-200:]))
print("trim:", repr(ex["text_trim"][-200:]))

texty bez ukončenej vety (vyhodené vo variantoch trim, trim_len):
{'OR': 705, 'CG': 431}


raw   priemerne slov: OR 73.6, CG 61.3


norm  priemerne slov: OR 73.6, CG 61.8


trim  priemerne slov: OR 74.8, CG 56.7

raw:  'Love this!  Well made, sturdy, and very comfortable.  I love it!Very pretty'
norm: 'Love this! Well made, sturdy, and very comfortable. I love it! Very pretty'
trim: 'Love this! Well made, sturdy, and very comfortable. I love it!'


In [4]:
# Kontrola artefaktov po úpravách: podiel recenzií s danou vlastnosťou
CHECKS = {
    "končí interpunkciou": END_RE.pattern,
    "spojenie viet bez medzery": r"[a-z0-9)][.!?][A-Z]",
    "dvojitá medzera": r"  ",
    "nový riadok": r"\n",
}
rows = {}
for v, col in [("raw", "text_raw"), ("norm", "text_norm"), ("trim", "text_trim")]:
    for y, name in [(0, "OR"), (1, "CG")]:
        s = df.loc[df["y"] == y, col].dropna()
        rows[(v, name)] = {k: s.str.contains(p, regex=True).mean() for k, p in CHECKS.items()}
display(pd.DataFrame(rows).round(3))

raw          norm        trim     
                              OR     CG     OR     CG   OR   CG
končí interpunkciou        0.906  0.352  0.906  0.352  1.0  1.0
spojenie viet bez medzery  0.000  0.285  0.000  0.000  0.0  0.0
dvojitá medzera            0.371  0.453  0.000  0.000  0.0  0.0
nový riadok                0.207  0.165  0.000  0.000  0.0  0.0

## 2. Rozdelenie a zarovnanie dĺžky
- stratifikované rozdelenie podľa kategórie × label pre každý seed (seed 42 = 02)
- `trim_len`: pásma dĺžky = kvantily počtu slov textu `trim` (všetky dáta); v každej časti (train / val / test) a v každom pásme sa väčšia trieda podvzorkuje na veľkosť menšej -> rovnaké rozdelenie dĺžky v oboch triedach

In [5]:
from sklearn.model_selection import train_test_split

def split_col(seed):
    strat = df["cat"] + "_" + df["y"].astype(str)
    trval, test = train_test_split(df.index, test_size=TEST_FRAC, stratify=strat, random_state=seed)
    train, val = train_test_split(trval, test_size=VAL_FRAC / (1 - TEST_FRAC), stratify=strat[trval], random_state=seed)
    s = pd.Series("train", index=df.index)
    s[val], s[test] = "val", "test"
    return s

for seed in SEEDS:
    df[f"split_{seed}"] = split_col(seed)

df["n_words_trim"] = df["text_trim"].str.split().str.len()
edges = np.unique(df["n_words_trim"].dropna().quantile(np.linspace(0, 1, LEN_BINS + 1)).values)
df["len_bin"] = pd.cut(df["n_words_trim"], edges, include_lowest=True, labels=False)
print("hranice pásiem (slová):", edges.astype(int).tolist())

def length_match(d, seed):
    keep = []
    for _, g in d.groupby("len_bin"):
        n = g["y"].value_counts()
        if len(n) < 2:
            continue
        m = n.min()
        keep += [g[g["y"] == y].sample(m, random_state=seed) for y in (0, 1)]
    return pd.concat(keep)

TEXT_COL = {"raw": "text_raw", "norm": "text_norm", "trim": "text_trim", "trim_len": "text_trim"}

def get_rows(variant, mask, seed):
    d = df[mask]
    if variant in ("trim", "trim_len"):
        d = d[d["text_trim"].notna()]
    if variant == "trim_len":
        d = length_match(d, seed)
    return d

def get_data(variant, mask, seed):
    d = get_rows(variant, mask, seed)
    return d[TEXT_COL[variant]].values, d["y"].values

for v in VARIANTS:
    sc = df[f"split_{SEEDS[0]}"]
    X, y = get_data(v, sc == "test", SEEDS[0])
    lens = pd.Series([len(t.split()) for t in X])
    print(f"{v:9s} test: {len(y):5d} riadkov, podiel CG {y.mean():.3f}, "
          f"medián slov OR {lens[y == 0].median():.0f} / CG {lens[y == 1].median():.0f}")

hranice pásiem (slová): [1, 10, 12, 14, 16, 18, 21, 24, 28, 32, 37, 43, 50, 59, 70, 84, 104, 131, 172, 233, 373]
raw       test:  6062 riadkov, podiel CG 0.500, medián slov OR 41 / CG 36
norm      test:  6062 riadkov, podiel CG 0.500, medián slov OR 41 / CG 36
trim      test:  5923 riadkov, podiel CG 0.502, medián slov OR 42 / CG 31
trim_len  test:  4764 riadkov, podiel CG 0.500, medián slov OR 42 / CG 41


## 3. Modely a pomocné funkcie
- `word`: slovné 1–2-gramy (ako 02); `char`: znakové 3–5-gramy (`char_wb`)
- `art`: `HistGradientBoostingClassifier` len z formálnych príznakov (ako 02, sekcia 4)
- prah z validácie (maximum F1); metriky: ROC-AUC, accuracy a F1 pri prahu z validácie, recall pri FPR 1 %
- pri vyvážených triedach je PR-AUC porovnateľné medzi variantmi; ROC-AUC je hlavná metrika

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, average_precision_score, roc_auc_score, roc_curve,
                             precision_recall_curve, precision_recall_fscore_support)

def artifact_features(texts):
    s = pd.Series(texts, dtype=str)
    n_w = s.str.split().str.len()
    n_sent = s.str.count(r"[.!?]+") + 1
    return pd.DataFrame({
        "n_words": n_w, "n_chars": s.str.len(),
        "ends_punct": s.str.contains(END_RE).astype(int),
        "n_joins": s.str.count(CHECKS["spojenie viet bez medzery"]),
        "n_double_space": s.str.count("  "), "n_newline": s.str.count("\n"),
        "n_digits": s.str.count(r"\d"), "n_ellipsis": s.str.count(r"\.\.\."), "n_excl": s.str.count("!"),
        "upper_ratio": s.str.count(r"[A-Z]") / s.str.len().clip(lower=1),
        "mean_word_len": s.str.replace(r"\s+", "", regex=True).str.len() / n_w.clip(lower=1),
        "words_per_sent": n_w / n_sent,
    })

class ArtModel:
    def fit(self, X, y):
        self.m = HistGradientBoostingClassifier(max_iter=300, random_state=0).fit(artifact_features(X), y)
        return self
    def predict_proba(self, X):
        return self.m.predict_proba(artifact_features(X))

class TfidfLR:
    def __init__(self, kind):
        if kind == "word":
            self.vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True)
        else:
            self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=5,
                                       sublinear_tf=True, max_features=300_000)
        self.clf = LogisticRegression(max_iter=2000)
    def fit(self, X, y):
        self.clf.fit(self.vec.fit_transform(X), y); return self
    def predict_proba(self, X):
        return self.clf.predict_proba(self.vec.transform(X))

def make_model(kind):
    return ArtModel() if kind == "art" else TfidfLR(kind)

def best_threshold(y, p):
    pr, rc, th = precision_recall_curve(y, p)
    f1 = 2 * pr * rc / np.maximum(pr + rc, 1e-12)
    return th[np.argmax(f1[:-1])]

def recall_at_fpr(y, p, fpr_max=TARGET_FPR):
    fpr, tpr, _ = roc_curve(y, p)
    return tpr[fpr <= fpr_max].max()

def evaluate(y, p, thr):
    pr, rc, f1, _ = precision_recall_fscore_support(y, p >= thr, average="binary", zero_division=0)
    return dict(roc_auc=roc_auc_score(y, p), pr_auc=average_precision_score(y, p),
                accuracy=accuracy_score(y, p >= thr), f1=f1, precision=pr, recall=rc,
                recall_at_fpr=recall_at_fpr(y, p), thr=thr, n_test=len(y), base_rate=y.mean())

def fit_eval(kind, variant, seed, train_mask, val_mask, test_mask):
    Xtr, ytr = get_data(variant, train_mask, seed)
    Xva, yva = get_data(variant, val_mask, seed)
    Xte, yte = get_data(variant, test_mask, seed)
    m = make_model(kind).fit(Xtr, ytr)
    thr = best_threshold(yva, m.predict_proba(Xva)[:, 1])
    p = m.predict_proba(Xte)[:, 1]
    return m, evaluate(yte, p, thr), (Xte, yte, p)

def mean_std(d, index, columns, value="roc_auc"):
    g = d.groupby(index + columns)[value]
    m = g.mean().unstack(columns).round(3)
    s = g.std().fillna(0).unstack(columns).round(3)
    return m.astype(str) + " ± " + s.astype(str)

## 4. Náhodné rozdelenie: varianty × modely × seedy
- 4 varianty × 3 modely (`art`, `word`, `char`) × 3 seedy = 36 modelov

In [7]:
rows, t0 = [], time.time()
total = len(VARIANTS) * 3 * len(SEEDS); n = 0
for v in VARIANTS:
    for kind in ["art", "word", "char"]:
        for seed in SEEDS:
            sc = df[f"split_{seed}"]
            _, r, _ = fit_eval(kind, v, seed, sc == "train", sc == "val", sc == "test")
            rows.append(dict(experiment="random", variant=v, kind=kind, seed=seed, test="all", **r))
            n += 1
            print(f"\r{n}/{total} modelov, {time.time() - t0:.0f} s", end="")
print()
resA = pd.DataFrame(rows)
resA["variant"] = pd.Categorical(resA["variant"], VARIANTS, ordered=True)

for metric, name in [("roc_auc", "ROC-AUC"), ("accuracy", "Accuracy (prah z validácie)"),
                     ("recall_at_fpr", f"Recall pri FPR {TARGET_FPR:.0%}")]:
    print(f"\n{name} (riadky = variant, stĺpce = model):")
    display(mean_std(resA, ["variant"], ["kind"], metric))

1/36 modelov, 2 s

2/36 modelov, 4 s

3/36 modelov, 6 s

4/36 modelov, 12 s

5/36 modelov, 18 s

6/36 modelov, 23 s

... (31 outputs left out)


Accuracy (prah z validácie) (riadky = variant, stĺpce = model):


kind,art,char,word
variant,,,
raw,0.871 ± 0.003,0.919 ± 0.001,0.929 ± 0.002
norm,0.854 ± 0.003,0.915 ± 0.001,0.929 ± 0.002
trim,0.765 ± 0.004,0.886 ± 0.004,0.901 ± 0.002
trim_len,0.738 ± 0.004,0.898 ± 0.004,0.914 ± 0.003



Recall pri FPR 1% (riadky = variant, stĺpce = model):


kind,art,char,word
variant,,,
raw,0.59 ± 0.008,0.692 ± 0.007,0.769 ± 0.038
norm,0.474 ± 0.027,0.671 ± 0.027,0.769 ± 0.038
trim,0.215 ± 0.003,0.575 ± 0.045,0.661 ± 0.029
trim_len,0.259 ± 0.025,0.636 ± 0.021,0.728 ± 0.039


## 5. Leave-one-category-out
- train: 9 kategórií (train časť seedu 42), prah z ich val časti; test: celá vynechaná kategória
- porovnanie s náhodným rozdelením -> pokles pri neznámej doméne

In [8]:
resB = None
if RUN_LOCO:
    sc = df[f"split_{LOCO_SEED}"]
    jobs = [(v, k) for v in VARIANTS for k in ["art", "word"]] + [(v, "char") for v in LOCO_CHAR_VARIANTS]
    rows, t0, n = [], time.time(), 0
    for held in CATS:
        other = df["cat"] != held
        for v, kind in jobs:
            _, r, _ = fit_eval(kind, v, LOCO_SEED, other & (sc == "train"), other & (sc == "val"), ~other)
            rows.append(dict(experiment="loco", variant=v, kind=kind, seed=LOCO_SEED, test=held, **r))
            n += 1
            print(f"\r{n}/{len(CATS) * len(jobs)} modelov, {time.time() - t0:.0f} s", end="")
    print()
    resB = pd.DataFrame(rows)
    resB["variant"] = pd.Categorical(resB["variant"], VARIANTS, ordered=True)

    print("ROC-AUC na vynechanej kategórii, priemer ± std cez kategórie:")
    display(mean_std(resB, ["variant"], ["kind"], "roc_auc"))
    print("\nword, ROC-AUC podľa vynechanej kategórie:")
    display(resB[resB["kind"] == "word"].pivot(index="test", columns="variant", values="roc_auc").round(3))

    rA = resA.groupby(["variant", "kind"], observed=True)["roc_auc"].mean()
    rB = resB.groupby(["variant", "kind"], observed=True)["roc_auc"].mean()
    print("\nPorovnanie ROC-AUC: náhodné rozdelenie vs. LOCO")
    display(pd.DataFrame({"náhodné": rA, "LOCO": rB, "rozdiel": rB - rA}).dropna().round(3))

1/100 modelov, 2 s

2/100 modelov, 8 s

3/100 modelov, 11 s

4/100 modelov, 17 s

5/100 modelov, 19 s

6/100 modelov, 26 s

... (95 outputs left out)


word, ROC-AUC podľa vynechanej kategórie:


variant,raw,norm,trim,trim_len
test,,,,
Books,0.970,0.970,0.958,0.965
Clothing_Shoes_and_Jewelry,0.960,0.960,0.947,0.948
Electronics,0.977,0.977,0.963,0.968
Home_and_Kitchen,0.982,0.982,0.960,0.966
Kindle_Store,0.971,0.971,0.966,0.970
Movies_and_TV,0.961,0.961,0.949,0.956
Pet_Supplies,0.971,0.971,0.951,0.956
Sports_and_Outdoors,0.980,0.980,0.960,0.968
Tools_and_Home_Improvement,0.979,0.979,0.963,0.967



Porovnanie ROC-AUC: náhodné rozdelenie vs. LOCO


náhodné   LOCO  rozdiel
variant  kind                         
raw      art     0.948  0.949    0.001
         char    0.974  0.965   -0.009
         word    0.980  0.972   -0.008
norm     art     0.935  0.934   -0.001
         word    0.980  0.972   -0.008
trim     art     0.856  0.856    0.001
         word    0.967  0.957   -0.011
trim_len art     0.843  0.845    0.002
         char    0.963  0.951   -0.012
         word    0.972  0.961   -0.010

## 6. Top príznaky
- LR natrénovaná na train časti seedu 42
- `raw` vs. `trim_len`; kladné koeficienty -> CG, záporné -> OR

In [9]:
def top_features(model, k=TOP_K):
    names, w = model.vec.get_feature_names_out(), model.clf.coef_[0]
    o = np.argsort(w)
    pos, neg = o[::-1][:k], o[:k]
    return pd.DataFrame({"-> CG": [repr(names[i]) for i in pos], "váha": w[pos].round(2),
                         "-> OR": [repr(names[i]) for i in neg], "váha ": w[neg].round(2)})

sc = df[f"split_{SEEDS[0]}"]
models = {}
for kind in ["word", "char"]:
    for v in ["raw", "trim_len"]:
        m, _, out = fit_eval(kind, v, SEEDS[0], sc == "train", sc == "val", sc == "test")
        models[(kind, v)] = (m, out)
        print(f"\n=== {kind}, {v}")
        display(top_features(m))


=== word, raw


,-> CG,váha,-> OR,váha
0,'the only',7.05,'at',-5.32
1,'the',6.62,'or',-4.92
2,'and the',6.14,'even',-4.71
3,'will keep',5.89,'from',-4.44
4,'and it',5.81,'all',-4.21
5,'that it',4.34,'on',-4.09
6,'the story',4.16,'when',-4.04
7,'is',4.08,'but',-3.99
8,'wide',3.93,'no',-3.97
9,'had',3.80,'after',-3.94



=== word, trim_len


,-> CG,váha,-> OR,váha
0,'and the',6.25,'at',-4.98
1,'and it',6.10,'or',-4.66
2,'the only',5.51,'even',-4.36
3,'will keep',4.32,'but',-4.07
4,'the story',4.12,'up',-3.79
5,'the',3.96,'though',-3.75
6,'but it',3.68,'on',-3.72
7,'had',3.63,'from',-3.66
8,'that it',3.42,'in',-3.61
9,'has the',3.26,'all',-3.61



=== char, raw


,-> CG,váha,-> OR,váha
0,' a ',8.73,' at',-3.37
1,'.i ',6.16,' at ',-3.18
2,'the ',6.06,'...',-2.93
3,' the ',6.03,' all ',-2.65
4,' is ',5.88,' all',-2.57
5,'he ',5.76,' - ',-2.43
6,' is',5.74,'ly ',-2.39
7,' i ',5.06,'.. ',-2.38
8,' the',4.32,'er ',-2.35
9,'the',4.04,' as',-2.34



=== char, trim_len


,-> CG,váha,-> OR,váha
0,' a ',6.12,' at',-3.14
1,' the ',4.93,' at ',-3.08
2,'the ',4.93,'...',-2.95
3,' is',4.83,'es ',-2.40
4,' is ',4.83,'.. ',-2.39
5,'he ',4.79,' or ',-2.38
6,' i ',3.89,'ly ',-2.36
7,' the',3.22,' . ',-2.30
8,' an ',3.15,'er ',-2.19
9,'the',2.83,'eve',-2.12


## 7. False negatives a false positives
- `word`, `trim_len`, seed 42, test časť
- false negatives: CG s najnižším skóre; false positives: OR s najvyšším skóre

In [10]:
m, (X, y, p) = models[("word", "trim_len")]
d = pd.DataFrame({"skóre": p.round(3), "label": y.astype(int), "text": X})
print("False negatives (CG, najnižšie skóre):")
display(d[d["label"] == 1].nsmallest(N_EXAMPLES, "skóre"))
print("False positives (OR, najvyššie skóre):")
display(d[d["label"] == 0].nlargest(N_EXAMPLES, "skóre"))

False negatives (CG, najnižšie skóre):


,skóre,label,text
147,0.043,1,"LESS THEN 1 YEAR OLD, BUT NO MORE!!"
357,0.079,1,"It's cute, but better for a costume than a night out."
193,0.081,1,Got as a joke for this.
136,0.088,1,DOES THE JOB WITH NO HURRY!!
129,0.093,1,"Easy to break off roll and assemble, as well."


False positives (OR, najvyššie skóre):


,skóre,label,text
810,0.975,0,"I have a lot of Maglite's this is the first ""C"" cell one and it works great."
693,0.960,0,My son is happy with it a very comfortable fit and the quality is durable to.
467,0.937,0,"My son loves everything that is Pokmon, and the movie was great for family."
1071,0.928,0,"This shoe is very nice. The fit is great, I have a wide foot & they fit & feel wonderful!"
1051,0.884,0,Got this for my daughters schooling and we also have the movie and they are both great items to have.


## 8. Uloženie výsledkov

In [11]:
RESULTS.mkdir(exist_ok=True)
out = pd.concat([resA] + ([resB] if resB is not None else []), ignore_index=True)
out.to_csv(RESULTS / "frd_baseline_02b.csv", index=False)
print("uložené:", RESULTS / "frd_baseline_02b.csv", out.shape)

uložené: ../../results/frd_baseline_02b.csv (136, 15)


## Výsledky zhrnutie
- varianty textu: `norm` odstráni spojenia viet, dvojité medzery a nové riadky (0 % v oboch triedach); `trim` zarovná koniec textu (100 % končí interpunkciou), vyhodí 1 136 textov bez ukončenej vety (705 OR, 431 CG), ale CG je po orezaní ešte kratšie (medián 31 vs. 42 slov); `trim_len` zarovná dĺžku (medián 41 vs. 42), test 4 764 riadkov
- **model len z artefaktov** (ROC-AUC, náhodné rozdelenie): raw 0,948 -> norm 0,935 -> trim 0,856 -> trim_len 0,843 -> úpravy odstránia väčšinu formálneho signálu, ale nie celý; zvyšok sú štylistické vlastnosti (priemerná dĺžka slova, dĺžka vety, `!`)
- **slovný TF-IDF + LR** (ROC-AUC / accuracy): raw 0,980 / 0,929, norm 0,980 / 0,929, trim 0,967 / 0,901, trim_len 0,972 / 0,914; std cez seedy ≤ 0,002
  - `norm` slovný model neovplyvní (tokenizér medzery ignoruje), `trim` zníži ROC-AUC o ~0,013 -> časť signálu baseline je z orezaného konca CG
  - recall pri FPR 1 %: 0,769 -> 0,728 (trim_len); vysoká variabilita (std 0,03–0,04)
- znakový model je vo všetkých variantoch mierne horší ako slovný (raw 0,974, trim_len 0,963); v `raw` využíva artefakty (`'.i '`, `'!very'` = spojené vety)
- **LOCO** (vynechaná kategória): word raw 0,972, trim_len 0,961; pokles oproti náhodnému rozdeleniu len 0,008–0,012 -> generalizácia na neznámu kategóriu je dobrá; najslabšie Clothing_Shoes_and_Jewelry a Toys_and_Games (0,948–0,950 pri trim_len)
- top príznaky (word aj char, aj po trim_len): CG -> časté funkčné slová a frázy (`the`, `and the`, `and it`, `the only`, `will keep`, `the story`, `' a '`, `' is '`); OR -> predložky a spojky (`at`, `or`, `even`, `though`, `after`, `when`), `...`, `!!`, `' - '` -> model zachytáva jednoduchší, opakujúci sa jazyk GPT-2, nie tému
- false negatives: krátke CG texty bez typických fráz, často celé veľkými písmenami (`DOES THE JOB WITH NO HURRY!!`)
- false positives: krátke, jednoduché pravé recenzie (`My son is happy with it a very comfortable fit...`)
- porovnanie s FraudSquad: FRD je pre baseline ľahší (ROC-AUC ~0,97 vs. FraudSquad recall@FPR ~0,93 pri inom podiele spamu), ale generátor (GPT-2 / ULMFiT, 2019) je oveľa slabší ako moderné LLM a dataset obsahuje silné artefakty generovania -> výsledky na `raw` FRD nadhodnocujú schopnosť detegovať generovaný text